# Product Data Cleaning & Feature Engineering

This notebook implements the data cleaning and feature engineering pipeline for the Verbalist product catalog based on the EDA findings.

## 1. Load the real CSV

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

BASE_DIR = Path().resolve().parent
RAW_DATA_PATH = BASE_DIR / "raw" / "dataset.csv"
OUTPUT_DATA_PATH = BASE_DIR / "processed" / "products_cleaned.csv"

df = pd.read_csv(RAW_DATA_PATH)
initial_row_count = len(df)
print(f"Loaded {initial_row_count} rows.")


Loaded 25768 rows.


## 2. Remove exact duplicate rows

In [2]:
df = df.drop_duplicates()
dedup_row_count = len(df)
duplicates_removed = initial_row_count - dedup_row_count
print(f"Removed {duplicates_removed} exact duplicate rows. Remaining: {dedup_row_count}")


Removed 63 exact duplicate rows. Remaining: 25705


## 3. Clean product names

In [3]:
def clean_name(name):
    if pd.isna(name): return ""
    name = str(name).strip()
    name = re.sub(r'\s+', ' ', name)
    return name

df['name'] = df['Product Name'].apply(clean_name)


## 4. Clean categories
Mapping promotional tags to 'other' and normalizing casing.

In [4]:
def normalize_category(cat):
    if pd.isna(cat): return 'other'
    cat = str(cat).lower()
    
    if any(x in cat for x in ['fruit', 'veg']): return 'fruits_vegetables'
    if any(x in cat for x in ['milk', 'dairy', 'cheese', 'butter', 'ghee', 'paneer']): return 'dairy'
    if any(x in cat for x in ['beverage', 'drink', 'coffee', 'tea', 'juice']): return 'beverages'
    if any(x in cat for x in ['snack', 'chips', 'biscuit', 'chocolate', 'namkeen']): return 'snacks'
    if any(x in cat for x in ['bakery', 'bread', 'cake']): return 'bakery'
    if any(x in cat for x in ['pantry', 'dal', 'pulse', 'rice', 'flour', 'oil', 'spice', 'sugar', 'salt', 'masala']): return 'pantry'
    if any(x in cat for x in ['personal', 'care', 'beauty', 'skin', 'hair', 'shaving', 'soap', 'shampoo']): return 'personal_care'
    if any(x in cat for x in ['household', 'clean', 'detergent', 'wash']): return 'household'
    if any(x in cat for x in ['baby', 'diaper', 'wipes']): return 'baby_care'
    if any(x in cat for x in ['frozen', 'ice cream']): return 'frozen'
    if any(x in cat for x in ['meat', 'fish', 'chicken', 'seafood']): return 'meat_seafood'
    if any(x in cat for x in ['stationery', 'pen', 'paper', 'book']): return 'stationery'
    
    return 'other'

df['category'] = df['Category'].apply(normalize_category)


## 5. Parse Quantity
Extract value and unit.

In [5]:
def parse_quantity(q_str):
    if pd.isna(q_str): return None, None
    q_str = str(q_str).strip().lower()
    match = re.search(r'^([\d\.]+)\s*([a-zA-Z]+)', q_str)
    if match:
        try:
            val = float(match.group(1))
            unit = match.group(2)
            return val, unit
        except:
            pass
    return None, None

parsed = df['Quantity'].apply(parse_quantity)
df['quantity_value'] = parsed.apply(lambda x: x[0])
df['quantity_unit'] = parsed.apply(lambda x: x[1])
parsed_success = df['quantity_value'].notna().sum()
parsed_failures = df['quantity_value'].isna().sum()


## 6 & 7. Clean price fields and detect sales

In [6]:
df['price'] = pd.to_numeric(df['Original Price (Rs.)'], errors='coerce')
df['sale_price'] = pd.to_numeric(df['Discounted Price (Rs.)'], errors='coerce')
df['price_available'] = df['price'].notna()

# Sale detection
df['is_on_sale'] = np.where(df['price_available'] & df['sale_price'].notna() & (df['sale_price'] < df['price']), True, False)


## 8, 9 & 10. Currency, Brand, Organic Detection

In [7]:
df['currency'] = 'INR'

def extract_brand(name):
    return None # Safely defaulting to None as per instructions

df['brand'] = df['name'].apply(extract_brand)

df['is_organic'] = df['name'].str.lower().str.contains('organic').fillna(False)


## 11. Search Aliases

In [8]:
def get_aliases(name):
    if not name: return "{}"
    aliases = [name.lower()]
    tokens = [t for t in name.lower().split() if len(t) > 3]
    aliases.extend(tokens)
    aliases = list(set(aliases))
    return "{" + ",".join(['"' + a.replace('"', '\\"') + '"' for a in aliases]) + "}"
    
df['search_aliases'] = df['name'].apply(get_aliases)


## 12, 13 & 14. Description, Availability, Source

In [9]:
df['description'] = None
df['is_available'] = True # Catalog availability default
df['source'] = 'bigbasket_dataset' # Meaningful dataset source


## 15. Schema Mapping

In [10]:
df['image_url'] = None

schema_cols = [
    'name', 'brand', 'category', 'description',
    'quantity_value', 'quantity_unit', 'price',
    'sale_price', 'currency', 'is_on_sale',
    'is_organic', 'is_available', 'search_aliases',
    'image_url', 'source'
]

out_df = df[schema_cols].copy()

# Extra cleanup for completely identical rows post-parsing
out_df = out_df.drop_duplicates()
final_row_count = len(out_df)


## 16. Data Quality Validation

In [11]:
val_failures = []
if out_df['name'].isna().sum() > 0: val_failures.append("Null names")
if out_df['category'].isna().sum() > 0: val_failures.append("Null categories")
if (out_df['quantity_value'] < 0).sum() > 0: val_failures.append("Negative quantities")
if (out_df['price'] < 0).sum() > 0: val_failures.append("Negative prices")
if (out_df['sale_price'] < 0).sum() > 0: val_failures.append("Negative sale prices")
if (out_df['sale_price'] > out_df['price']).sum() > 0: val_failures.append("Sale price > price")
if out_df['is_on_sale'].isna().sum() > 0: val_failures.append("is_on_sale has nulls")
if out_df.duplicated().sum() > 0: val_failures.append("Exact duplicates remain")
print(f"Validation failures: {val_failures}")


Validation failures: []


## 17. Summary

In [12]:
print("--- Cleaning Summary ---")
print(f"Original row count: {initial_row_count}")
print(f"Duplicate rows removed: {initial_row_count - final_row_count}")
print(f"Final row count: {final_row_count}")
print(f"Missing prices: {out_df['price'].isna().sum()}")
print(f"Number of products on sale: {out_df['is_on_sale'].sum()}")
print(f"Number of organic products: {out_df['is_organic'].sum()}")
print(f"Unique categories: {out_df['category'].nunique()}")
print(f"Successfully parsed quantities: {out_df['quantity_value'].notna().sum()}")


--- Cleaning Summary ---
Original row count: 25768
Duplicate rows removed: 1352
Final row count: 24416
Missing prices: 2541
Number of products on sale: 14984
Number of organic products: 523
Unique categories: 9
Successfully parsed quantities: 23442


## 18. Export

In [13]:
out_df.to_csv(OUTPUT_DATA_PATH, index=False)
print(f"Exported {final_row_count} rows to {OUTPUT_DATA_PATH}")


Exported 24416 rows to C:\Users\namde\OneDrive\Desktop\VERBALIST\data\processed\products_cleaned.csv
